In [1]:
import pandas as pd
import numpy as np

In [2]:
arrivals = 'UN_Tourism_inbound_arrivals_12_2025.xlsx' #extract total inbound arrivals & tourists (overnight visitors)
purpose = 'UN_Tourism_inbound_arrivals_by_purpose_12_2025.xlsx' #extract business and personal arrivals
transport = 'UN_Tourism_inbound_arrivals_by_transport_12_2025.xlsx' #extract data for air arrivals
acc_establishments = 'UN_Tourism_accommodation_hotels_12_2025.xlsx' #extract info on establishments, capacity, and average length of stay
expenditure = 'UN_Tourism_inbound_expenditure_12_2025.xlsx' #extract expenditure data
inbound_acc = 'UN_Tourism_inbound_accommodation_12_2025.xlsx' #extract data for guests hosted and number of overnights, also use it to calculate average length of stay in case we have missing values

In [3]:
def clean_tourism(file_path, indicator_keywords):
    """
    Reads a UN Tourism Excel file, filters for relevant indicators, 
    and returns a cleaned dataframe.
    """
    try:
        df = pd.read_excel(file_path, sheet_name='Data')
        #standardize column names
        df.columns = [c.lower().strip() for c in df.columns]
        
        #filter for the most relevant 'Total' indicators to avoid double counting
        pattern = '|'.join(indicator_keywords)
        df_filtered = df[df['indicator_label'].str.contains(pattern, case=False, na=False)].copy()
        
        #focus on the most recent 6 years of data for trend analysis (2019-2024)
        recent_years = sorted(df_filtered['year'].unique())[-6:]
        df_filtered = df_filtered[df_filtered['year'].isin(recent_years)]
        
        #select and rename core columns
        df_clean = df_filtered[['reporter_area_label', 'year', 'value', 'unit', 'indicator_label']]
        return df_clean
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return pd.DataFrame()

In [4]:
#process arrivals extract total arrivals data
df_arrivals = clean_tourism(arrivals, ['tourists', 'total - visitors'])
df_arrivals.head(5)

,reporter_area_label,year,value,unit,indicator_label
1892,Albania,2019,6128.0,thousand trips,inbound - trips - total - total - overnight vi...
1893,Albania,2020,2604.0,thousand trips,inbound - trips - total - total - overnight vi...
1894,Albania,2021,5515.0,thousand trips,inbound - trips - total - total - overnight vi...
1895,Albania,2022,7104.7,thousand trips,inbound - trips - total - total - overnight vi...
1896,Albania,2023,9741.0,thousand trips,inbound - trips - total - total - overnight vi...


In [5]:
#values are in thousand trips so need to convert to absolute
df_arrivals['value']= df_arrivals['value']*1000

In [6]:
#separate overnight vs. total visitors
#find out unique values in indicator label column
df_arrivals['indicator_label'].dropna().unique()

<ArrowStringArray>
['inbound - trips - total - total - overnight visitors (tourists)', 'inbound - trips - total - total - visitors']
Length: 2, dtype: str

In [7]:
#seperate the arrivals values into two columns
df_arrivals = df_arrivals.pivot_table(
        index=['reporter_area_label', 'year'],
        columns='indicator_label',
        values = 'value',
        aggfunc='sum').reset_index()

df_arrivals.head(5)

indicator_label,reporter_area_label,year,inbound - trips - total - total - overnight visitors (tourists),inbound - trips - total - total - visitors
0,Albania,2019,6128000.0,6406000.0
1,Albania,2020,2604000.0,2658000.0
2,Albania,2021,5515000.0,5689000.0
3,Albania,2022,7104700.0,7543800.0
4,Albania,2023,9741000.0,10156000.0


In [8]:
#rename columns
df_arrivals.rename(columns = {
    'inbound - trips - total - total - overnight visitors (tourists)': 'tourists',
    'inbound - trips - total - total - visitors': 'total_arrivals'},
                   inplace=True)

df_arrivals.head(5)

indicator_label,reporter_area_label,year,tourists,total_arrivals
0,Albania,2019,6128000.0,6406000.0
1,Albania,2020,2604000.0,2658000.0
2,Albania,2021,5515000.0,5689000.0
3,Albania,2022,7104700.0,7543800.0
4,Albania,2023,9741000.0,10156000.0


In [9]:
#process travel by purpose to extract business and personal arrivals
#process purpose file
#will use this to extract business and personal (leisure) arrivals
df_purpose = clean_tourism(purpose, ['tourists'])
df_purpose.head(5)

,reporter_area_label,year,value,unit,indicator_label
21,Albania,2019,101.2,thousand trips,inbound - trips - by purpose - business - over...
22,Albania,2020,115.3,thousand trips,inbound - trips - by purpose - business - over...
23,Albania,2021,100.2,thousand trips,inbound - trips - by purpose - business - over...
24,Albania,2022,106.3,thousand trips,inbound - trips - by purpose - business - over...
41,Algeria,2019,158.1,thousand trips,inbound - trips - by purpose - business - over...


In [10]:
#values are in thousand trips so need to convert to absolute
df_purpose['value']= df_purpose['value']*1000
df_purpose = df_purpose[['reporter_area_label', 'year', 'value', 'indicator_label']]
df_purpose.head(5)

,reporter_area_label,year,value,indicator_label
21,Albania,2019,101200.0,inbound - trips - by purpose - business - over...
22,Albania,2020,115300.0,inbound - trips - by purpose - business - over...
23,Albania,2021,100200.0,inbound - trips - by purpose - business - over...
24,Albania,2022,106300.0,inbound - trips - by purpose - business - over...
41,Algeria,2019,158100.0,inbound - trips - by purpose - business - over...


In [11]:
#need to seperate the arrivals data and organize it so we have a column for business arrivals, personal arrivals, and total arrivals
#find out unique values in indicator label column
df_purpose['indicator_label'].dropna().unique()

<ArrowStringArray>
['inbound - trips - by purpose - business - overnight visitors (tourists)',
 'inbound - trips - by purpose - personal - overnight visitors (tourists)',
    'inbound - trips - by purpose - total - overnight visitors (tourists)']
Length: 3, dtype: str

In [12]:
#pivot the data into 3 columns
#business_visitors, personal_visitors, total_visitors
df_purpose = df_purpose.pivot_table(
        index=['reporter_area_label', 'year'],
        columns='indicator_label',
        values = 'value',
        aggfunc='sum').reset_index()

df_purpose.head(5)

indicator_label,reporter_area_label,year,inbound - trips - by purpose - business - overnight visitors (tourists),inbound - trips - by purpose - personal - overnight visitors (tourists),inbound - trips - by purpose - total - overnight visitors (tourists)
0,Albania,2019,101200.0,6304800.0,6406000.0
1,Albania,2020,115300.0,2542500.0,2657800.0
2,Albania,2021,100200.0,5588400.0,5688600.0
3,Albania,2022,106300.0,7437500.0,7543800.0
4,Algeria,2019,158100.0,1775700.0,1933800.0


In [13]:
#rename columns
df_purpose.rename(columns = {
    'inbound - trips - by purpose - business - overnight visitors (tourists)': 'business_arrivals',
    'inbound - trips - by purpose - personal - overnight visitors (tourists)': 'personal_arrivals',
    'inbound - trips - by purpose - total - overnight visitors (tourists)': 'total_arrivals'},
                   inplace=True)

#clean travel purpose data
df_purpose = df_purpose[['reporter_area_label', 'year', 'business_arrivals', 'personal_arrivals']]
df_purpose.head(5)

indicator_label,reporter_area_label,year,business_arrivals,personal_arrivals
0,Albania,2019,101200.0,6304800.0
1,Albania,2020,115300.0,2542500.0
2,Albania,2021,100200.0,5588400.0
3,Albania,2022,106300.0,7437500.0
4,Algeria,2019,158100.0,1775700.0


In [14]:
#pull air transport arrivals data
df_transport = clean_tourism(transport, ['air'])
df_transport.head(5)

,reporter_area_label,year,value,unit,indicator_label
24,Albania,2019,783.9,thousand trips,inbound - trips - transport - air - overnight ...
25,Albania,2020,269.8,thousand trips,inbound - trips - transport - air - overnight ...
26,Albania,2021,764.7,thousand trips,inbound - trips - transport - air - overnight ...
27,Albania,2022,1250.0,thousand trips,inbound - trips - transport - air - overnight ...
70,Angola,2019,218.0,thousand trips,inbound - trips - transport - air - overnight ...


In [15]:
#rename value column 
df_transport.rename(columns={'value': 'air_arrivals'}, inplace=True)
#values in thousand trips need to be converted to absolute
df_transport['air_arrivals'] = df_transport['air_arrivals']*1000

#clean transport data
df_transport = df_transport[['reporter_area_label', 'year', 'air_arrivals']]

df_transport.head(5)

,reporter_area_label,year,air_arrivals
24,Albania,2019,783900.0
25,Albania,2020,269800.0
26,Albania,2021,764700.0
27,Albania,2022,1250000.0
70,Angola,2019,218000.0


In [16]:
#process data on accommodation establishments, capacity and average length of stay
df_acc_establishments = clean_tourism(acc_establishments, ['establishments', 'length of stay', 'bed', 'rooms'])
df_acc_establishments.head()

,reporter_area_label,year,value,unit,indicator_label
1,Albania,2019,1126.0,units,capacity - accommodation - short-term accommod...
2,Albania,2020,1237.0,units,capacity - accommodation - short-term accommod...
3,Albania,2021,1256.0,units,capacity - accommodation - short-term accommod...
4,Albania,2022,1385.0,units,capacity - accommodation - short-term accommod...
5,Albania,2023,1549.0,units,capacity - accommodation - short-term accommod...


In [17]:
#need to seperate establishments and length of stay data into different columns
#unique values in indicator label column
df_acc_establishments['indicator_label'].dropna().unique()

<ArrowStringArray>
[          'capacity - accommodation - short-term accommodation, hotels and similar (ISIC 5510) - establishments',
               'capacity - accommodation - short-term accommodation, hotels and similar (ISIC 5510) - bed places',
                    'capacity - accommodation - short-term accommodation, hotels and similar (ISIC 5510) - rooms',
        'capacity - accommodation - short-term accommodation, hotels and similar (ISIC 5510) - bed places - rate',
 'capacity - accommodation - short-term accommodation, hotels and similar (ISIC 5510) - length of stay - average',
             'capacity - accommodation - short-term accommodation, hotels and similar (ISIC 5510) - rooms - rate']
Length: 6, dtype: str

In [18]:
#pivot the data into separate columns
#establishments, average length of stay, capacity
#for capacity, we'll add bed places and rooms for countries that have both
df_acc_establishments = df_acc_establishments.pivot_table(
        index=['reporter_area_label', 'year'],
        columns='indicator_label',
        values = 'value',
        aggfunc='sum').reset_index()

df_acc_establishments.head(5)

indicator_label,reporter_area_label,year,"capacity - accommodation - short-term accommodation, hotels and similar (ISIC 5510) - bed places","capacity - accommodation - short-term accommodation, hotels and similar (ISIC 5510) - bed places - rate","capacity - accommodation - short-term accommodation, hotels and similar (ISIC 5510) - establishments","capacity - accommodation - short-term accommodation, hotels and similar (ISIC 5510) - length of stay - average","capacity - accommodation - short-term accommodation, hotels and similar (ISIC 5510) - rooms","capacity - accommodation - short-term accommodation, hotels and similar (ISIC 5510) - rooms - rate"
0,Albania,2019,77974.0,NaN,1126.0,2.580000,33798.0,21.81
1,Albania,2020,82434.0,NaN,1237.0,2.344786,34713.0,10.59
2,Albania,2021,85403.0,NaN,1256.0,2.564628,35802.0,17.50
3,Albania,2022,96986.0,NaN,1385.0,2.413045,42464.0,18.90
4,Albania,2023,112551.0,NaN,1549.0,2.571192,50010.0,NaN


In [19]:
#create a clean capacity column
#for capacity, we'll add bed places and rooms for countries that have both
df_acc_establishments['acc_capacity'] = df_acc_establishments['capacity - accommodation - short-term accommodation, hotels and similar (ISIC 5510) - bed places'] + df_acc_establishments['capacity - accommodation - short-term accommodation, hotels and similar (ISIC 5510) - rooms']
#rename the other columns
df_acc_establishments.rename(columns={'capacity - accommodation - short-term accommodation, hotels and similar (ISIC 5510) - establishments': 'acc_establishments',
                                   'capacity - accommodation - short-term accommodation, hotels and similar (ISIC 5510) - length of stay - average': 'ALOS'},
                             inplace=True)

#clean acc_establishments data
df_accommodation = df_acc_establishments[['reporter_area_label', 'year', 'acc_establishments', 'acc_capacity', 'ALOS']]
df_accommodation.head()

indicator_label,reporter_area_label,year,acc_establishments,acc_capacity,ALOS
0,Albania,2019,1126.0,111772.0,2.580000
1,Albania,2020,1237.0,117147.0,2.344786
2,Albania,2021,1256.0,121205.0,2.564628
3,Albania,2022,1385.0,139450.0,2.413045
4,Albania,2023,1549.0,162561.0,2.571192


In [20]:
#pull expenditure data
df_expenditure = clean_tourism(expenditure, ['total'])
df_expenditure.head(5)

,reporter_area_label,year,value,unit,indicator_label
4022,Afghanistan,2019,85.000000,million US dollars (nominal values),inbound - expenditure - balance of payments - ...
4023,Afghanistan,2020,74.833144,million US dollars (nominal values),inbound - expenditure - balance of payments - ...
4048,Albania,2019,2458.000000,million US dollars (nominal values),inbound - expenditure - balance of payments - ...
4049,Albania,2020,1242.855632,million US dollars (nominal values),inbound - expenditure - balance of payments - ...
4050,Albania,2021,2479.950546,million US dollars (nominal values),inbound - expenditure - balance of payments - ...


In [21]:
#expenditure values are in millions so we need to convert them
df_expenditure['expenditure']=df_expenditure['value']*1000000
df_expenditure = df_expenditure[['reporter_area_label', 'year', 'expenditure']]
df_expenditure.head(5)

,reporter_area_label,year,expenditure
4022,Afghanistan,2019,8.500000e+07
4023,Afghanistan,2020,7.483314e+07
4048,Albania,2019,2.458000e+09
4049,Albania,2020,1.242856e+09
4050,Albania,2021,2.479951e+09


In [22]:
##merge the datasets on country and year
#combine arrivals and purpose data
merged_df = pd.merge(
    df_purpose, 
    df_arrivals, 
    on=['reporter_area_label', 'year'], 
    suffixes=('_arrivals', '_expenditure')
)
merged_df.head(5)

indicator_label,reporter_area_label,year,business_arrivals,personal_arrivals,tourists,total_arrivals
0,Albania,2019,101200.0,6304800.0,6128000.0,6406000.0
1,Albania,2020,115300.0,2542500.0,2604000.0,2658000.0
2,Albania,2021,100200.0,5588400.0,5515000.0,5689000.0
3,Albania,2022,106300.0,7437500.0,7104700.0,7543800.0
4,Algeria,2019,158100.0,1775700.0,NaN,2371000.0


In [23]:
#check for null values 
merged_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 578 entries, 0 to 577
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   reporter_area_label  578 non-null    str    
 1   year                 578 non-null    int64  
 2   business_arrivals    549 non-null    float64
 3   personal_arrivals    565 non-null    float64
 4   tourists             550 non-null    float64
 5   total_arrivals       486 non-null    float64
dtypes: float64(4), int64(1), str(1)
memory usage: 33.0 KB


In [24]:
#handle the missing values in business, personal and total arrivals
merged_df['business_arrivals'] = merged_df['business_arrivals'].fillna(merged_df['total_arrivals'] - merged_df['personal_arrivals'])
merged_df['personal_arrivals'] = merged_df['personal_arrivals'].fillna(merged_df['total_arrivals'] - merged_df['business_arrivals'])
merged_df['total_arrivals'] = merged_df['total_arrivals'].fillna(merged_df['personal_arrivals'] + merged_df['business_arrivals'])
merged_df.head()

indicator_label,reporter_area_label,year,business_arrivals,personal_arrivals,tourists,total_arrivals
0,Albania,2019,101200.0,6304800.0,6128000.0,6406000.0
1,Albania,2020,115300.0,2542500.0,2604000.0,2658000.0
2,Albania,2021,100200.0,5588400.0,5515000.0,5689000.0
3,Albania,2022,106300.0,7437500.0,7104700.0,7543800.0
4,Algeria,2019,158100.0,1775700.0,NaN,2371000.0


In [25]:
#merge air arrivals data 
merged_df = pd.merge(merged_df, df_transport, on=['reporter_area_label', 'year'], how='left')
merged_df.head()

,reporter_area_label,year,business_arrivals,personal_arrivals,tourists,total_arrivals,air_arrivals
0,Albania,2019,101200.0,6304800.0,6128000.0,6406000.0,783900.0
1,Albania,2020,115300.0,2542500.0,2604000.0,2658000.0,269800.0
2,Albania,2021,100200.0,5588400.0,5515000.0,5689000.0,764700.0
3,Albania,2022,106300.0,7437500.0,7104700.0,7543800.0,1250000.0
4,Algeria,2019,158100.0,1775700.0,NaN,2371000.0,NaN


In [26]:
#merge expenditure data
merged_df = pd.merge(merged_df, df_expenditure, on=['reporter_area_label', 'year'], how='left')
merged_df.head()

,reporter_area_label,year,business_arrivals,personal_arrivals,tourists,total_arrivals,air_arrivals,expenditure
0,Albania,2019,101200.0,6304800.0,6128000.0,6406000.0,783900.0,2.458000e+09
1,Albania,2020,115300.0,2542500.0,2604000.0,2658000.0,269800.0,1.242856e+09
2,Albania,2021,100200.0,5588400.0,5515000.0,5689000.0,764700.0,2.479951e+09
3,Albania,2022,106300.0,7437500.0,7104700.0,7543800.0,1250000.0,3.255061e+09
4,Algeria,2019,158100.0,1775700.0,NaN,2371000.0,NaN,1.400000e+08


In [27]:
#further merge accommodation data
merged_df = pd.merge(merged_df, df_accommodation,on=['reporter_area_label', 'year'], how='left')

#compute expenditure per arrival 
merged_df['exp_per_arrival'] = merged_df['expenditure'] / merged_df['total_arrivals']

merged_df.head(5)

,reporter_area_label,year,business_arrivals,personal_arrivals,tourists,total_arrivals,air_arrivals,expenditure,acc_establishments,acc_capacity,ALOS,exp_per_arrival
0,Albania,2019,101200.0,6304800.0,6128000.0,6406000.0,783900.0,2.458000e+09,1126.0,111772.0,2.580000,383.702779
1,Albania,2020,115300.0,2542500.0,2604000.0,2658000.0,269800.0,1.242856e+09,1237.0,117147.0,2.344786,467.590531
2,Albania,2021,100200.0,5588400.0,5515000.0,5689000.0,764700.0,2.479951e+09,1256.0,121205.0,2.564628,435.920293
3,Albania,2022,106300.0,7437500.0,7104700.0,7543800.0,1250000.0,3.255061e+09,1385.0,139450.0,2.413045,431.488289
4,Algeria,2019,158100.0,1775700.0,NaN,2371000.0,NaN,1.400000e+08,1417.0,NaN,1.680000,59.046816


In [28]:
#double check guests and overnights
#import inbound guests and number of overnights from inbound acc. data
#extract data for guests hosted and number of overnights
#also use it to calculate average length of stay to use as replacement for ALOS missing values
inbound_acc = 'UN_Tourism_inbound_accommodation_12_2025.xlsx' 
#process data on accommodation establishments, capacity and average length of stay
df_inbound_acc = clean_tourism(inbound_acc, ['hotels'])
df_inbound_acc.head()

,reporter_area_label,year,value,unit,indicator_label
3863,Albania,2019,736.000,thousand guests,inbound - accommodation - short-term accommoda...
3864,Albania,2020,280.514,thousand guests,inbound - accommodation - short-term accommoda...
3865,Albania,2021,656.500,thousand guests,inbound - accommodation - short-term accommoda...
3866,Albania,2022,846.359,thousand guests,inbound - accommodation - short-term accommoda...
3867,Albania,2023,1305.194,thousand guests,inbound - accommodation - short-term accommoda...


In [29]:
#values in thousand guests and thousand overnights
df_inbound_acc['value'] = df_inbound_acc['value']*1000

In [30]:
#need to seperate number of guests and overnight stays into different columns
#unique values in indicator label column
df_inbound_acc['indicator_label'].dropna().unique()

<ArrowStringArray>
['inbound - accommodation - short-term accommodation, hotels and similar (ISIC 5510) - guests', 'inbound - accommodation - short-term accommodation, hotels and similar (ISIC 5510) - overnights']
Length: 2, dtype: str

In [31]:
#pivot the data into separate columns
#guests and overnights
df_inbound_acc = df_inbound_acc.pivot_table(
        index=['reporter_area_label', 'year'],
        columns='indicator_label',
        values = 'value',
        aggfunc='sum').reset_index()

df_inbound_acc.head(5)

indicator_label,reporter_area_label,year,"inbound - accommodation - short-term accommodation, hotels and similar (ISIC 5510) - guests","inbound - accommodation - short-term accommodation, hotels and similar (ISIC 5510) - overnights"
0,Albania,2019,736000.0,1987000.0
1,Albania,2020,280514.0,748363.0
2,Albania,2021,656500.0,1932096.0
3,Albania,2022,846359.0,2358400.0
4,Albania,2023,1305194.0,3639857.0


In [32]:
#rename the guest and overnight columns
df_inbound_acc.rename(columns={'inbound - accommodation - short-term accommodation, hotels and similar (ISIC 5510) - guests': 'acc_guests',
                                   'inbound - accommodation - short-term accommodation, hotels and similar (ISIC 5510) - overnights': 'acc_overnights'},
                             inplace=True)

#clean df_inbound_acc data
df_inbound_acc = df_inbound_acc[['reporter_area_label', 'year', 'acc_guests', 'acc_overnights']]
df_inbound_acc.head()

indicator_label,reporter_area_label,year,acc_guests,acc_overnights
0,Albania,2019,736000.0,1987000.0
1,Albania,2020,280514.0,748363.0
2,Albania,2021,656500.0,1932096.0
3,Albania,2022,846359.0,2358400.0
4,Albania,2023,1305194.0,3639857.0


In [33]:
#merge inbound_acc to the main dataset
merged_df = pd.merge(merged_df, df_inbound_acc, on=['reporter_area_label', 'year'], how='left')
merged_df.head()

,reporter_area_label,year,business_arrivals,personal_arrivals,tourists,total_arrivals,air_arrivals,expenditure,acc_establishments,acc_capacity,ALOS,exp_per_arrival,acc_guests,acc_overnights
0,Albania,2019,101200.0,6304800.0,6128000.0,6406000.0,783900.0,2.458000e+09,1126.0,111772.0,2.580000,383.702779,736000.0,1987000.0
1,Albania,2020,115300.0,2542500.0,2604000.0,2658000.0,269800.0,1.242856e+09,1237.0,117147.0,2.344786,467.590531,280514.0,748363.0
2,Albania,2021,100200.0,5588400.0,5515000.0,5689000.0,764700.0,2.479951e+09,1256.0,121205.0,2.564628,435.920293,656500.0,1932096.0
3,Albania,2022,106300.0,7437500.0,7104700.0,7543800.0,1250000.0,3.255061e+09,1385.0,139450.0,2.413045,431.488289,846359.0,2358400.0
4,Algeria,2019,158100.0,1775700.0,NaN,2371000.0,NaN,1.400000e+08,1417.0,NaN,1.680000,59.046816,851000.0,1401000.0


In [34]:
#handle missing values in average length of stay, guests, overnights
merged_df['ALOS'] = merged_df['ALOS'].fillna(round((merged_df['acc_overnights']/merged_df['acc_guests']),2))
merged_df['acc_guests'] = merged_df['acc_guests'].fillna(round((merged_df['acc_overnights'] *merged_df['acc_guests']),2))
merged_df['acc_overnights'] = merged_df['acc_overnights'].fillna(round((merged_df['ALOS'] *merged_df['acc_guests']),2))
merged_df.head()

,reporter_area_label,year,business_arrivals,personal_arrivals,tourists,total_arrivals,air_arrivals,expenditure,acc_establishments,acc_capacity,ALOS,exp_per_arrival,acc_guests,acc_overnights
0,Albania,2019,101200.0,6304800.0,6128000.0,6406000.0,783900.0,2.458000e+09,1126.0,111772.0,2.580000,383.702779,736000.0,1987000.0
1,Albania,2020,115300.0,2542500.0,2604000.0,2658000.0,269800.0,1.242856e+09,1237.0,117147.0,2.344786,467.590531,280514.0,748363.0
2,Albania,2021,100200.0,5588400.0,5515000.0,5689000.0,764700.0,2.479951e+09,1256.0,121205.0,2.564628,435.920293,656500.0,1932096.0
3,Albania,2022,106300.0,7437500.0,7104700.0,7543800.0,1250000.0,3.255061e+09,1385.0,139450.0,2.413045,431.488289,846359.0,2358400.0
4,Algeria,2019,158100.0,1775700.0,NaN,2371000.0,NaN,1.400000e+08,1417.0,NaN,1.680000,59.046816,851000.0,1401000.0


In [35]:
#output dataset to csv
merged_df.to_csv('inbound_clean.csv', index=False)